In [ ]:
"""
Third-order cumulant closure for the driven-dissipative Kerr oscillator
========================================================================

    H(t) = omega_0 a+a + eps*cos(wd t)*(a+a+) + (chi/2)(a+)^2 a^2
    L    = sqrt(kappa) a

This script derives the closed ODE system for
    mu          = <alpha>                       (complex mean)
    sigma^2     = <delta_alpha delta_alpha*>     (real)
    tilde_s2    = <delta_alpha^2>                (complex, squeezing)
    kappa_30    = <delta_alpha^3>                (complex, 3rd cumulant)
    kappa_21    = <delta_alpha^2 delta_alpha*>   (complex, 3rd cumulant)
i.e. 9 real coupled ODEs, closing the Gaussian ansatz at 3rd cumulant
order (cumulants of order >= 4 set to zero).

METHOD (four exact steps + one approximation, clearly isolated):

  1. The Husimi Q-function obeys (derived in Husimi_harmonic_balance.tex):
         dQ/dt = d/da(A Q) + d/da*(A* Q) + kappa d^2Q/(da da*)
                 + (i chi/2)[a^2 d^2Q/da^2 - a*^2 d^2Q/da*^2]
     with A = i*Delta*a + i*eps_t + i*chi*a^2*a* + kappa/2*a.
     For ANY polynomial f(a,a*), integrating by parts gives the EXACT
     moment identity:
         d<f>/dt = -<A df/da> - <A* df/da*> + kappa<d^2f/(da da*)>
                   + (i chi/2)[<d^2(a^2 f)/da^2> - <d^2(a*^2 f)/da*^2>]
     Setting f = a^m a*^n gives an exact recursion for raw moments R_mn.
     EXACT -- no truncation here.

  2. Raw moments are related to central moments by the ordinary binomial
     expansion alpha = mu + delta_alpha. EXACT -- no truncation here.

  3. Central moments are related to cumulants by the standard
     moment-cumulant formula (via the cumulant generating function,
     exp(K(s,u))). THE ONLY APPROXIMATION: cumulants of order >= 4 are
     set to zero.

  4. Central-moment ODEs are extracted from the raw-moment ODEs by
     product-rule back-substitution, solved order by order (mu first,
     then sigma^2/tilde_s2, then kappa_30/kappa_21) -- a triangular
     linear system, EXACT given steps 1-3.

Every step below prints a check. In particular Section 5 verifies that
setting kappa_30 = kappa_21 = 0 reproduces the tex's boxed Gaussian-order
(2nd order) equations EXACTLY -- the crucial consistency check.
"""

import sympy as sp
I = sp.I

# ======================================================================
# 0. Symbols
# ======================================================================
Delta, chi, kap, eps, wd, t = sp.symbols('Delta chi kappa epsilon omega_d t', real=True)

# The 9 real dynamical variables, split into real/imaginary parts so
# sympy never has to guess about conjugation -- conjugates are just
# sign flips on the *_i symbols, so there is no risk of the kind of
# conjugation slip that bit the two-mode SNAIL derivation.
mu_r, mu_i   = sp.symbols('mu_r mu_i', real=True)
s2           = sp.symbols('s2', real=True)                 # sigma^2 (provably real, checked below)
ts2_r, ts2_i = sp.symbols('ts2_r ts2_i', real=True)         # tilde sigma^2
k30_r, k30_i = sp.symbols('k30_r k30_i', real=True)         # kappa_30
k21_r, k21_i = sp.symbols('k21_r k21_i', real=True)         # kappa_21

mu,  mus  = mu_r + I*mu_i,   mu_r - I*mu_i
ts2, ts2s = ts2_r + I*ts2_i, ts2_r - I*ts2_i
k30, k30s = k30_r + I*k30_i, k30_r - I*k30_i
k21, k21s = k21_r + I*k21_i, k21_r - I*k21_i


# ======================================================================
# 1. Exact FPE moment recursion (no truncation in this section)
# ======================================================================
a, ac = sp.symbols('a ac')     # formal symbols standing for alpha, alpha*

# Drive kept exactly, including the counter-rotating piece at 2*omega_d
# (this is what makes the NESS genuinely periodic, not static).
eps_t  = eps/2*(1 + sp.exp( 2*I*wd*t))
eps_tc = eps/2*(1 + sp.exp(-2*I*wd*t))

A  =  I*Delta*a  + I*eps_t  + I*chi*a**2*ac  + kap/2*a
Ac = -I*Delta*ac - I*eps_tc - I*chi*ac**2*a  + kap/2*ac

def moment_rhs_monomials(m, n):
    """
    Exact d<a^m ac^n>/dt, returned as {(m',n'): coeff} meaning
    coeff * <a^m' ac^n'>. Straight from the FPE moment identity, no
    approximation.
    """
    f = a**m * ac**n
    expr = (
        -A  * sp.diff(f, a)
        -Ac * sp.diff(f, ac)
        + kap * sp.diff(f, a, ac)
        + I*chi/2 * (sp.diff(a**2*f, a, 2) - sp.diff(ac**2*f, ac, 2))
    )
    poly = sp.Poly(sp.expand(expr), a, ac)
    return {monom: coeff for monom, coeff in poly.terms()}

print("=" * 70)
print("STEP 1: exact recursion, checked against the tex's own hand derivation")
print("=" * 70)
print("d<alpha>/dt monomials:     ", moment_rhs_monomials(1, 0))
print("  (tex: R10 coeff = -(i*Delta+kappa/2-2*i*chi); R21 coeff = -i*chi)")
print("d<alpha^2>/dt monomials:   ", moment_rhs_monomials(2, 0))
print("d<|alpha|^2>/dt monomials: ", moment_rhs_monomials(1, 1))
print("  (tex: chi-term / R22 coefficient must be ABSENT/zero for m=n)")
print()


# ======================================================================
# 2+3. Moment <-> cumulant closure at 3rd order (steps 2 and 3 combined)
# ======================================================================
# Joint cumulants kappa_{jk} of (delta_alpha, delta_alpha*).
# kappa_{10}=kappa_{01}=0 automatically (these are CENTRAL moments).
# THE ONLY APPROXIMATION IN THIS SCRIPT: cumulants of order >=4 -> 0.
kappa = {
    (2, 0): ts2, (1, 1): s2,  (0, 2): ts2s,
    (3, 0): k30, (2, 1): k21, (1, 2): k21s, (0, 3): k30s,
}

s, u = sp.symbols('s u')   # formal generating variables (s<->delta_alpha, u<->delta_alpha*)

K2 = kappa[(2, 0)]*s**2/2 + kappa[(1, 1)]*s*u + kappa[(0, 2)]*u**2/2
K3 = kappa[(3, 0)]*s**3/6 + kappa[(2, 1)]*s**2*u/2 + kappa[(1, 2)]*s*u**2/2 + kappa[(0, 3)]*u**3/6

# exp(K2+K3) truncated to total degree <= 5. Since K2 is EXACTLY degree 2
# and K3 EXACTLY degree 3, exp(K)=sum K^n/n! only needs n=0,1,2 to reach
# degree 5 (K3^2, degree 6, and all n>=3 terms are dropped) -- this is
# exact truncation bookkeeping, not a second approximation.
G = sp.expand(1 + K2 + K3 + sp.Rational(1, 2)*K2**2 + K2*K3)

def M(j, k):
    """Central moment <delta_alpha^j delta_alpha*^k>, order j+k <= 5."""
    if j == 0 and k == 0:
        return sp.Integer(1)
    return sp.expand(sp.diff(G, s, j, u, k).subs({s: 0, u: 0}))

Mvals = {(j, k): M(j, k) for j in range(6) for k in range(6) if 0 < j + k <= 5}

def R(m, n):
    """
    Raw moment <alpha^m alpha*^n>, exact binomial expansion around mu:
        <alpha^m alpha*^n> = sum_{j,k} C(m,j) C(n,k) mu^(m-j) mu*^(n-k) M_jk
    """
    total = 0
    for j in range(m + 1):
        for k in range(n + 1):
            Mjk = sp.Integer(1) if (j == 0 and k == 0) else Mvals.get((j, k), sp.Integer(0))
            total += sp.binomial(m, j) * sp.binomial(n, k) * mu**(m-j) * mus**(n-k) * Mjk
    return sp.expand(total)

print("=" * 70)
print("STEP 2+3: closure check -- order-4 moments should be UNCHANGED by")
print("kappa_30/kappa_21 (a 4-object set only partitions as 2+2, not 2+3)")
print("=" * 70)
print("M(3,1) [expect pure Gaussian 3*sigma^2*tilde_s2]:          ", sp.expand(M(3, 1)))
print("M(2,2) [expect pure Gaussian 2*sigma^4 + |tilde_s2|^2]:    ", sp.expand(M(2, 2)))
print("M(4,1) [order 5 -- SHOULD depend on kappa30/21, via 2+3]:")
print("   ", sp.expand(M(4, 1)))
print()


# ======================================================================
# 4. Full dot-R_mn (recursion output substituted into central moments)
# ======================================================================
def dotR(m, n):
    monomials = moment_rhs_monomials(m, n)
    return sp.expand(sum(coeff * R(mm, nn) for (mm, nn), coeff in monomials.items()))

# Triangular back-substitution (each line uses only already-known dots):
#   R10 = mu                                     => dmu  = dotR(1,0)
#   R11 = mu*mu* + s2                             => ds2  = dotR(1,1) - dmu*mu* - mu*dmu*
#   R20 = mu^2 + ts2                              => dts2 = dotR(2,0) - 2*mu*dmu
#   R30 = mu^3 + 3*mu*ts2 + k30                   => dk30 = dotR(3,0) - 3*mu^2*dmu - 3*dmu*ts2 - 3*mu*dts2
#   R21 = mu^2*mu* + mu**ts2 + 2*mu*s2 + k21      => dk21 = dotR(2,1) - (...)
dR10, dR01 = dotR(1, 0), dotR(0, 1)
dmu, dmus  = dR10, dR01

dR11 = dotR(1, 1)
ds2  = sp.expand(dR11 - dmu*mus - mu*dmus)

dR20 = dotR(2, 0)
dts2 = sp.expand(dR20 - 2*mu*dmu)

dR30 = dotR(3, 0)
dk30 = sp.expand(dR30 - 3*mu**2*dmu - 3*dmu*ts2 - 3*mu*dts2)

dR21 = dotR(2, 1)
dk21 = sp.expand(dR21 - (2*mu*mus*dmu + mu**2*dmus) - (dmus*ts2 + mus*dts2) - (2*dmu*s2 + 2*mu*ds2))

print("=" * 70)
print("STEP 4: internal consistency checks")
print("=" * 70)
print("dR01 == conjugate(dR10)?               ", sp.simplify(dR01 - dR10.conjugate()) == 0)
print("Im(ds2) == 0 (sigma^2 stays real)?     ", sp.simplify(sp.im(sp.expand(ds2))) == 0)
print()


# ======================================================================
# 5. THE critical validation: kappa30, kappa21 -> 0 must reproduce the
#    tex's boxed Gaussian-order equations EXACTLY.
# ======================================================================
zero_3rd = {k30_r: 0, k30_i: 0, k21_r: 0, k21_i: 0}

tex_dmu  = -I*eps_t - (I*Delta + kap/2 - 2*I*chi)*mu - I*chi*(mu**2*mus + 2*mu*s2 + mus*ts2)
tex_ds2  = kap*(1 - s2) - 2*chi*sp.im(mus**2*ts2)
tex_dts2 = -(2*I*Delta + kap - 5*I*chi + 4*I*chi*mu*mus + 6*I*chi*s2)*ts2 + I*chi*mu**2*(1 - 2*s2)

print("=" * 70)
print("STEP 5: reduction to Gaussian order (the key validation)")
print("=" * 70)
print("dmu  matches tex exactly?  ", sp.simplify(dmu.subs(zero_3rd)  - tex_dmu)  == 0)
print("ds2  matches tex exactly?  ", sp.simplify(ds2.subs(zero_3rd)  - tex_ds2)  == 0)
print("dts2 matches tex exactly?  ", sp.simplify(dts2.subs(zero_3rd) - tex_dts2) == 0)
print()


# ======================================================================
# 6. Recombine into clean complex notation (purely cosmetic -- undoes
#    the real/imag split that kept step 0-5 safe from conjugation bugs)
# ======================================================================
Mu, Mus, Ts2, Ts2s, K30, K30s, K21, K21s = sp.symbols(
    'Mu Mus Ts2 Ts2s K30 K30s K21 K21s')
S2 = sp.Symbol('S2', real=True)

sub = {
    mu_r: (Mu + Mus)/2,    mu_i: -I*(Mu - Mus)/2,
    s2:   S2,
    ts2_r: (Ts2 + Ts2s)/2, ts2_i: -I*(Ts2 - Ts2s)/2,
    k30_r: (K30 + K30s)/2, k30_i: -I*(K30 - K30s)/2,
    k21_r: (K21 + K21s)/2, k21_i: -I*(K21 - K21s)/2,
}

def recombine(expr):
    return sp.expand(sp.simplify(sp.expand(expr.subs(sub))))

print("=" * 70)
print("FINAL EQUATIONS (complex notation: Mu=mu, Mus=mu*, S2=sigma^2,")
print("Ts2=tilde-sigma^2, Ts2s=tilde-sigma^2*, K30=kappa_30, K21=kappa_21, etc.)")
print("=" * 70)
print("\ndot(mu)       =", recombine(dmu))
print("\ndot(sigma^2)  =", recombine(ds2))
print("\ndot(tilde_s2) =", recombine(dts2))
print("\ndot(kappa30)  =", recombine(dk30))
print("\ndot(kappa21)  =", recombine(dk21))

# Keep the raw (real/imag-split) sympy expressions around for anyone who
# wants to lambdify them directly for numerics (see companion script).
RESULTS = dict(
    variables=dict(mu_r=mu_r, mu_i=mu_i, s2=s2, ts2_r=ts2_r, ts2_i=ts2_i,
                   k30_r=k30_r, k30_i=k30_i, k21_r=k21_r, k21_i=k21_i),
    params=dict(Delta=Delta, chi=chi, kap=kap, eps=eps, wd=wd, t=t),
    odes=dict(dmu=dmu, ds2=ds2, dts2=dts2, dk30=dk30, dk21=dk21),
)

if __name__ == "__main__":
    pass

In [ ]:
RESULTS["odes"]["dmu"]